# Multimodal RAG — Chatting With a PDF That Has Pictures In It

**Course:** Text & Language Processing / LLM Practical Track
**Format:** Bonus notebook — off the numbered track, more playing than grading
**Suggested duration:** 60–90 minutes
**Prerequisite:** Tutorial 3 (LangChain), sections 7–8. You should already have built a text-only RAG chain.

Tutorial 3 built RAG over a list of tidy Python strings. Real documents are not tidy Python strings.
They are PDFs, and a PDF is a bag of ink positions that *happens* to look like a document. The most
useful part of a paper is often the one part a plain text loader turns into gibberish: the figure, the
results table, the architecture diagram everybody screenshots.

This notebook fixes that. By the end you will have a chain you can ask *"what does the encoder stack
actually look like?"* and get an answer **grounded in Figure 1 of the Transformer paper** — because the
figure itself was retrieved and handed to a model with eyes.

The trick at the centre of it is delightfully cheap, and you can state it in one line:

> **You cannot embed a picture with a text embedder. So embed a *description* of the picture, and keep
> the picture itself on a shelf with a numbered ticket.**

That is the whole idea. Everything below is plumbing.

*Based on Alejandro AO's walkthrough [Multimodal RAG: Chat with PDFs (Images & Tables)](https://www.youtube.com/watch?v=uLrReyH5cu0),
ported to the stack this course already uses — Groq, Chroma, and local embeddings.*


## Why text-only RAG fails quietly

Run an ordinary PDF loader over a research paper and watch what happens to the interesting parts:

| What is on the page | What a text loader hands you |
|---|---|
| A results table with eight columns | `28.4 41.8 2.3 · 10^19 38.1 ...` — numbers with no columns |
| An architecture diagram | nothing at all, or the labels in arbitrary order |
| A bar chart | the caption, and only the caption |
| A two-column layout | left and right columns interleaved line by line |

The failure is **silent**, which is the dangerous part. Your chain still answers. It just answers from
the model's memory of a famous paper instead of from the paper you gave it, and you have built
something that looks grounded and is not. Tutorial 3 made that point about prompts; here it returns as
a parsing problem.

So we need three things a character splitter cannot do:

1. **Layout awareness** — know that *this* blob is a table and *that* one is a figure.
2. **A fitting representation per type** — tables as HTML so the columns survive, figures as images.
3. **A model that can look at an image at answer time**, not merely at index time.


## Setup, and the two system libraries that will try to ruin your afternoon

The Python packages are unremarkable:

```bash
pip install "unstructured[pdf]" pillow pypdf pymupdf
pip install langchain-core langchain-groq langchain-chroma langchain-huggingface sentence-transformers python-dotenv
```

Now the awkward part. `unstructured` in `hi_res` mode shells out to **two programs that are not Python
packages**:

- **poppler** — renders PDF pages to images so a layout model can look at them
- **tesseract** — OCR, for anything that is ink rather than text

Install them the way your OS prefers:

```bash
# Windows, via conda  (easiest — do this one)
conda install -c conda-forge poppler tesseract

# Windows, via chocolatey
choco install poppler tesseract

# macOS
brew install poppler tesseract

# Linux / Google Colab
sudo apt-get install -y poppler-utils tesseract-ocr
```

**If that sounds like a bad afternoon, skip it.** There is a pure-pip **Plan B** further down that
needs no system libraries at all. It reads layout less well, and the rest of the notebook does not
notice the difference.

Finally, the usual `.env` file next to this notebook:

```text
GROQ_API_KEY=your-key-here
```


In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

assert os.getenv("GROQ_API_KEY"), (
    "GROQ_API_KEY not found. Create a .env file next to this notebook "
    "containing: GROQ_API_KEY=your-key-here"
)

# Two models, two jobs. The summariser reads; the vision model looks.
TEXT_MODEL   = "llama-3.3-70b-versatile"   # fast and cheap, text only
VISION_MODEL = "qwen/qwen3.6-27b"          # accepts images as well as text

print("Key loaded.")
print("Summariser :", TEXT_MODEL)
print("Eyes       :", VISION_MODEL)


In [ ]:
import shutil

for exe, package in [("pdftoppm", "poppler"), ("tesseract", "tesseract-ocr")]:
    status = "found" if shutil.which(exe) else "MISSING  ->  hi_res will fail; use Plan B"
    print(f"{package:<16} {exe:<12} {status}")


### Code walkthrough — two models on purpose

**Why two.** Summarising forty text chunks with a vision model would be slow and pointless — they have
no pixels in them. `llama-3.3-70b-versatile` does the reading, and the vision model is held back for
the handful of things that are actually images. Routing work by *what the input is* rather than sending
everything to the largest model available is most of cost control in practice.

**Model names rot.** Groq retired the Llama 4 vision models in mid-2026, which is why this notebook
uses Qwen 3.6. If `VISION_MODEL` starts returning a 404, open
[console.groq.com/docs/vision](https://console.groq.com/docs/vision), take whatever is listed there and
change one string. Nothing else in the notebook depends on it. Treat every model name in every tutorial
you read — including this one — as a variable with a short shelf life.

**`shutil.which`** simply asks whether a binary is on your `PATH`. Checking now costs a second. Finding
out halfway through a six-minute parse costs six minutes.


## Our victim: *Attention Is All You Need*

Pleasingly self-referential: we are about to build a Transformer-powered system whose entire job is
answering questions about the paper that introduced Transformers. It is also an ideal test case —
Figure 1 is the architecture diagram, Figure 2 is the attention mechanism, and Tables 1–3 are dense
numeric results. Precisely the material text-only RAG drops on the floor.

We take **the first nine pages**, because `hi_res` runs a real layout model over every page on your CPU
and you would like this to finish while the coffee is still warm. Those nine pages hold both figures
and all three tables, so nothing interesting is lost.


In [ ]:
import pathlib
import urllib.request
from pypdf import PdfReader, PdfWriter

PDF_URL = "https://arxiv.org/pdf/1706.03762"
full_paper = pathlib.Path("attention_full.pdf")
paper      = pathlib.Path("attention_p1-9.pdf")

if not full_paper.exists():
    print("downloading...")
    urllib.request.urlretrieve(PDF_URL, full_paper)

if not paper.exists():
    reader, writer = PdfReader(str(full_paper)), PdfWriter()
    for page in reader.pages[:9]:
        writer.add_page(page)
    with paper.open("wb") as fh:
        writer.write(fh)

print(f"{paper}  -  {len(PdfReader(str(paper)).pages)} pages, {paper.stat().st_size / 1e6:.2f} MB")


## Shredding the PDF properly

This is the cell that does the real work, and it is the slow one: **expect two to six minutes.**
`hi_res` renders every page to an image and runs a document-layout detection model over it to find the
boxes — this blob is `NarrativeText`, that one is a `Table`, that one is an `Image`.

Start it, then read the walkthrough underneath while it grinds.


In [ ]:
from unstructured.partition.pdf import partition_pdf

chunks = partition_pdf(
    filename=str(paper),

    # --- how hard to look ---
    strategy="hi_res",                    # layout model + OCR. "fast" = text extraction only, no figures
    infer_table_structure=True,           # tables come back as HTML, columns intact

    # --- what to pull out as pixels ---
    extract_image_block_types=["Image"],  # crop every detected figure...
    extract_image_block_to_payload=True,  # ...as base64 in metadata, rather than files on disk

    # --- how to group the results ---
    chunking_strategy="by_title",         # start a new chunk at each section heading
    max_characters=8000,                  # hard ceiling on one chunk
    combine_text_under_n_chars=2000,      # glue runt sections onto their neighbour
    new_after_n_chars=6000,               # soft ceiling: prefer to break past this point
)

from collections import Counter
print(Counter(type(c).__name__ for c in chunks))


### Code walkthrough — the parameters that actually matter

**`strategy="hi_res"`** is the entire point of the exercise. The alternative, `"fast"`, pulls the PDF's
text layer and stops — which is exactly the failure mode we came here to avoid. `hi_res` costs minutes
of CPU and buys the ability to tell a figure from a paragraph.

**`infer_table_structure=True`** gives every table a `metadata.text_as_html` attribute: a real
`<table>` with real `<tr>` and `<td>`. This matters more than it sounds. `28.4 41.8 3.3` is three
numbers in a row; `<td>28.4</td><td>41.8</td>` says which column each number lives in. LLMs read HTML
tables remarkably well, so keeping the structure is accuracy you get for free.

**`extract_image_block_to_payload=True`** keeps each cropped figure in memory as base64 instead of
writing PNGs into a folder you then have to manage. Notebook-friendly — and base64 is exactly the
format the vision API wants later, so there is no conversion step at all.

**`chunking_strategy="by_title"`** runs a second pass that groups elements at section boundaries.
Semantically far better than the fixed-character splitting of tutorial 3: *3.2 Attention* stays in one
piece instead of being guillotined at character 1000.

**The three character limits** are one ceiling and two hints. `max_characters` is absolute.
`new_after_n_chars` says "once you are past this, take the next opportunity to break".
`combine_text_under_n_chars` merges stubs, so you do not index a chunk whose entire content is the word
*Conclusion*.

Now look at the `Counter` output. You have `CompositeElement` objects (grouped text) and `Table`
objects. Notice what is **missing**: no `Image`, despite our having asked for images. That is not a bug,
and the next section is about where they went.


## Three piles: text, tables, pictures

The images are hiding. When the chunker groups elements it keeps the originals it swallowed in
`chunk.metadata.orig_elements`, so a figure that sat inside section 3.2 is now *inside* the chunk for
section 3.2 rather than beside it. You have to go in after it.

Tables, by contrast, the chunker leaves alone — they come back as their own top-level elements, because
merging a table into a paragraph would destroy it.


In [ ]:
texts, tables, images = [], [], []

for chunk in chunks:
    kind = type(chunk).__name__

    if "Table" in kind:
        tables.append(chunk.metadata.text_as_html)

    elif "CompositeElement" in kind:
        texts.append(chunk.text)
        # the figures were swallowed by this chunk - dig them back out
        for element in chunk.metadata.orig_elements:
            if "Image" in type(element).__name__:
                images.append(element.metadata.image_base64)

print(f"{len(texts):>3} text chunks")
print(f"{len(tables):>3} tables")
print(f"{len(images):>3} images")


### Code walkthrough — three plain Python lists

Note what these three variables are by the end: **lists of strings**. `texts` holds plain text,
`tables` holds HTML, `images` holds base64. No library objects, no `Element` subclasses, nothing that
ties the rest of the notebook to `unstructured`.

That is deliberate, and it is a habit worth stealing. The parsing library is the most volatile part of
any document pipeline — you will swap it. Normalising to boring built-in types at the boundary means
the swap touches exactly one cell. Plan B, below, exploits precisely this: a completely different
parser, the same three lists, and not one line of downstream code changes.

**`type(chunk).__name__`** rather than `isinstance` is a small pragmatism: `unstructured` has both
`Table` and `TableChunk`, and matching on the name catches both without importing either.


### Plan B — no system libraries, no tears

Skip this if `hi_res` worked. If poppler and tesseract defeated you, run the cell below instead: it
uses **PyMuPDF**, which is a single pip install with no system dependencies, and produces the very same
three lists.

The trade is real and worth understanding. PyMuPDF has no layout model — it reads the PDF's own
internal structure. So it finds tables only when the PDF drew them with actual ruling lines, and it
extracts images only when they are embedded bitmaps. A figure drawn as vector graphics (which is most
diagrams in LaTeX papers, including Figure 1 of our victim) is invisible to it. You get a working
pipeline; you get fewer pictures.


In [ ]:
# ---- Plan B: run this ONLY if partition_pdf above failed ----
def extract_with_pymupdf(path):
    # Same three lists, pure pip, worse layout sense.
    import base64
    import fitz  # PyMuPDF

    texts, tables, images = [], [], []
    doc = fitz.open(path)

    for page in doc:
        page_text = page.get_text().strip()
        if page_text:
            texts.append(page_text)

        for table in page.find_tables():
            tables.append(table.to_pandas().to_html(index=False))

        for xref, *_ in page.get_images(full=True):
            pix = fitz.Pixmap(doc, xref)
            if pix.n - pix.alpha > 3:              # CMYK -> RGB, or PNG encoding refuses
                pix = fitz.Pixmap(fitz.csRGB, pix)
            images.append(base64.b64encode(pix.tobytes("png")).decode())

    return texts, tables, images


# texts, tables, images = extract_with_pymupdf(str(paper))
# print(len(texts), "text", len(tables), "tables", len(images), "images")


## Look at what fell out

The best part of this pipeline is that you can *see* it working. Let us put a table and a figure on the
screen exactly as the model will receive them.


In [ ]:
import base64
from IPython.display import HTML, Image, display

if tables:
    print("A table, rendered from the extracted HTML:")
    display(HTML(tables[0]))
else:
    print("No tables found.")

if images:
    print(f"\nA figure, decoded from base64 ({len(images[0]):,} characters of it):")
    display(Image(base64.b64decode(images[0])))
else:
    print("\nNo images found - if you used Plan B, this is expected for vector figures.")


### Code walkthrough — why this cell earns its place

It is not decoration. Looking at the extracted artefacts is the single highest-value debugging step in
any document pipeline, and skipping it is how people ship RAG systems that retrieve blank crops of
page margins for three weeks without noticing.

`display(HTML(...))` renders the table's HTML in the notebook, which tells you at a glance whether
`infer_table_structure` actually recovered the columns or produced one long row.

`base64.b64decode` turns the payload string back into PNG bytes, and `IPython.display.Image` renders
them. Look at the crop: is it the whole figure, or did the layout model clip the caption off? That crop
is literally what the vision model will see — there is no second, better copy anywhere.


## The trick: embed a rumour about the picture

Here is the problem in one sentence. Our embedding model, `all-MiniLM-L6-v2`, maps **text** to a vector.
Hand it a base64 string and it will cheerfully embed the *characters* `/9j/4AAQSkZJRg...`, producing a
vector that means nothing and matches nothing.

The classic workaround is a CLIP-style joint image–text embedder, which puts pictures and words in one
shared space. That works, and it requires a second embedding model, a second index, and a new set of
things to tune.

The cheap workaround — the one in this notebook — is this:

```text
                for each element
                       |
        +--------------+--------------+
        |                             |
   ask an LLM to             keep the original
   describe it                  untouched
        |                             |
   embed THAT                 store on a shelf
        |                             |
   vector store  <-- same doc_id -->  dictionary
```

Search runs against the descriptions, which are text, so an ordinary text embedder is all you need.
But what you *retrieve* is the original — the actual table, the actual image — because the description
carries a ticket number pointing at it.

The asymmetry is the elegant bit. **Descriptions are optimised for being found; originals are optimised
for being read.** They no longer have to be the same object, and the moment you stop insisting that
they are, a lot of awkward problems dissolve.


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq

summariser = ChatGroq(model=TEXT_MODEL, temperature=0)

summary_prompt = ChatPromptTemplate.from_template(
    "You are writing a search-index entry for one fragment of a research paper.\n"
    "Summarise it in two or three sentences. Name the concepts, numbers, model names and entities it "
    "mentions, so that somebody searching for any of them lands on this fragment.\n"
    "Reply with the summary only - no preamble, no 'Here is a summary'.\n\n"
    "Fragment:\n{element}"
)

summarise = summary_prompt | summariser | StrOutputParser()

text_summaries  = summarise.batch(texts,  {"max_concurrency": 3})
table_summaries = summarise.batch(tables, {"max_concurrency": 3})

print("ORIGINAL :", texts[1][:200].replace("\n", " "), "...")
print()
print("SUMMARY  :", text_summaries[1])


### Code walkthrough — the prompt is doing retrieval engineering

Read the prompt again. It does **not** say "summarise this". It says *name the concepts, numbers and
entities so somebody searching for them lands here*. That difference is the difference between a
summary written for a human and a summary written for a **cosine similarity function**.

A good literary summary — "this section discusses the model's efficiency characteristics" — is a
*terrible* index entry, because a student who asks "how many FLOPs did the big model need?" shares no
vocabulary with it. Keeping the numbers and the proper nouns in the summary keeps them in the
embedding, and the embedding is the only thing search ever sees. When multimodal RAG retrieves badly,
this prompt is the first place to look, not the embedding model.

**`"Reply with the summary only"`** matters more than it should. Without it, roughly a third of your
index entries begin with "Here is a concise summary of the fragment:" — eight tokens of pure noise
contributing to every single vector.

**`.batch(..., {"max_concurrency": 3})`** sends the requests in parallel but at most three at a time.
The concurrency cap is not politeness, it is self-defence: Groq's free tier will start returning 429s
if you fire forty requests at once, and `batch` does not retry them for you.

**`temperature=0`** because you want the same chunk to produce the same index entry every run.
Re-indexing should not silently change what your system can find.


In [ ]:
from langchain_core.messages import HumanMessage

# reasoning_effort="none" puts Qwen in non-thinking mode: we want a caption, not a soliloquy.
# If your langchain-groq is older and rejects that argument, use:
#     ChatGroq(model=VISION_MODEL, temperature=0, model_kwargs={"reasoning_effort": "none"})
eyes = ChatGroq(model=VISION_MODEL, temperature=0, reasoning_effort="none")

IMAGE_PROMPT = (
    "You are writing a search-index entry for a figure from a research paper.\n"
    "Describe what the figure shows in two or three sentences: its type (diagram, plot, table...), "
    "every label and axis name you can read, and the relationship or architecture it depicts. "
    "Transcribe any text visible in the image.\n"
    "Reply with the description only."
)


def describe_image(b64):
    message = HumanMessage(content=[
        {"type": "text", "text": IMAGE_PROMPT},
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
    ])
    return eyes.invoke([message]).content


image_summaries = [describe_image(b64) for b64 in images]

for i, summary in enumerate(image_summaries):
    print(f"[image {i}] {summary[:220]}\n")


### Code walkthrough — how you actually send a picture to an LLM

**Message content becomes a list.** Everywhere else in this course, `HumanMessage` took a string. A
multimodal message takes a *list of blocks* instead, each tagged with a `type`. Text blocks and image
blocks sit side by side in one message, and their order is meaningful — the model reads them in the
order you wrote them, so the instruction goes first and the picture second.

**`data:image/jpeg;base64,...`** is a data URI: the image inlined into the request rather than linked.
The API will equally accept a public `https://` URL there, but our crops exist only in memory, so
inlining is the only option — which is why `extract_image_block_to_payload=True` was the right call two
sections ago. `unstructured` emits JPEG; if you switch to Plan B's PyMuPDF path, its output is PNG, so
the prefix must become `data:image/png;base64,`.

**Images are not free.** On Qwen 3.6 each image costs about 2,048 input tokens regardless of size, and
a single request accepts at most five. Both numbers shape the design of the answering chain further
down, where we cap how many figures go into one prompt.

**`reasoning_effort="none"`** switches Qwen out of thinking mode. For "describe this picture" the
reasoning tokens are latency you pay for and never read.

**A plain loop, not `.batch`.** Vision calls are heavy and there are only a handful of images; running
them one at a time keeps you comfortably inside the rate limit and makes a failure easy to locate.


## Two stores, one ticket number

Now we assemble the shelf and the catalogue.

- **Chroma** holds the summaries, embedded. This is what search looks at.
- **A plain Python `dict`** holds the originals. This is what you get back.
- A **`doc_id`** in each summary's metadata joins them.

LangChain ships this arrangement as `MultiVectorRetriever`, and the original video uses it. We are
going to write it out by hand instead — partly because it is eight lines, partly because its import
path moved house in LangChain v1, and mostly because a retriever you can read is a retriever you can
debug.


In [ ]:
import uuid

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma(
    collection_name="attention_multimodal",
    embedding_function=embeddings,
)

docstore = {}          # doc_id -> the original text / table HTML / base64 image


def index(summaries, originals, kind):
    ids = [str(uuid.uuid4()) for _ in originals]

    vectorstore.add_documents([
        Document(page_content=summary, metadata={"doc_id": doc_id, "kind": kind})
        for summary, doc_id in zip(summaries, ids)
    ])
    docstore.update(zip(ids, originals))


index(text_summaries,  texts,  "text")
index(table_summaries, tables, "table")
index(image_summaries, images, "image")

print(f"{len(docstore)} originals on the shelf, {len(docstore)} summaries in the index")


In [ ]:
def retrieve(question, k=4):
    # Search the summaries, hand back the originals.
    hits = vectorstore.similarity_search(question, k=k)
    return [(hit.metadata["kind"], docstore[hit.metadata["doc_id"]]) for hit in hits]


for kind, original in retrieve("What does the Transformer architecture look like?"):
    preview = "<base64 image>" if kind == "image" else original[:110].replace("\n", " ")
    print(f"{kind:<6} | {preview}")


### Code walkthrough — the whole retriever, in two functions

**`index()`** mints a UUID per element, stores the *summary* in Chroma with that UUID in its metadata,
and files the *original* in the dict under the same key. Three calls, one per pile. Note that the pile
it came from is recorded as `kind` — the answering chain needs it to decide whether a retrieved item is
a string to paste or a picture to attach.

**`retrieve()`** is where the sleight of hand happens, and it is two lines. `similarity_search` compares
your question against the summaries and returns summary `Document`s. We then throw those summaries away
and use their `doc_id` to look up what they were describing. **The summaries exist only to be found.**
They never reach the model that answers.

Look at the output: at least one row should say `image`. A text-only embedder just retrieved a picture,
because it never had to understand the picture — only the sentence we wrote about it.

**`HuggingFaceEmbeddings` runs locally**, as in tutorial 3. Only the model calls leave your machine;
the embeddings are free and need no key.

**`Chroma(...)` with no `persist_directory`** keeps the index in memory, so it evaporates when the
kernel restarts. That is what you want while experimenting. For anything you would rather not re-parse,
pass `persist_directory="./db"` — and remember that only the vector store is persisted. `docstore` is a
dict, and dicts do not survive kernel restarts either; a real deployment puts the originals in Redis,
S3, or a database.


## Feeding the pictures back to the model

The last piece. Retrieval now returns a mixed bag — some strings, some base64 images — and the prompt
has to carry both: text pasted into the instruction, images attached as blocks.


In [ ]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

MAX_IMAGES = 3          # Qwen 3.6 accepts at most 5 per request; leave headroom


def build_prompt(payload):
    question = payload["question"]

    context_text, context_images = [], []
    for kind, original in payload["context"]:
        (context_images if kind == "image" else context_text).append(original)

    joined = "\n\n---\n\n".join(context_text)

    content = [{
        "type": "text",
        "text": (
            "Answer the question using only the context below, which may contain text, HTML tables "
            "and figures from a research paper. If the context does not contain the answer, say so - "
            "do not guess. When a figure or table is what supports your answer, say which one.\n\n"
            f"Context:\n{joined}\n\n"
            f"Question: {question}"
        ),
    }]

    for b64 in context_images[:MAX_IMAGES]:
        content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/jpeg;base64,{b64}"},
        })

    return [HumanMessage(content=content)]


chain = (
    {"context": RunnableLambda(retrieve), "question": RunnablePassthrough()}
    | RunnableLambda(build_prompt)
    | eyes
    | StrOutputParser()
)

print(chain.invoke("What are the two sub-layers in each encoder layer, and how are they connected?"))


### Code walkthrough — the same shape as tutorial 3, one step richer

The chain will look familiar, because it is structurally identical to the RAG chain in tutorial 3,
section 7:

```text
{context: retriever, question: passthrough}  ->  prompt  ->  model  ->  parser
```

The dict at the head is a `RunnableParallel` in shorthand: the question goes down two paths at once, to
the retriever and straight through untouched, and both land in the next step. The only thing that
changed is that `format_docs` — which flattened documents into a string — has become `build_prompt`,
which flattens *some* of them into a string and turns the rest into image blocks.

**The sort is the whole function.** One line splits the mixed bag by `kind`, and everything else is
assembly. Keeping the text and the images in one message rather than two is what lets the model relate
them — "the figure shows what the paragraph describes" is a connection it can only make if both arrive
together.

**`context_images[:MAX_IMAGES]`** is not optional. Exceed the per-request image limit and the API
rejects the whole call; each image also costs ~2,048 tokens, so an uncapped list is a good way to blow
the context window on a `k=8` retrieval. Truncating is the boring, correct answer.

**`eyes` answers, not `summariser`.** The final model must be the one that can see, since the prompt
now contains images. This is why the vision model earns its keep twice: once at index time to describe
figures, once at answer time to read them.


## Ask it something only a picture can answer

Now the real test. The three questions below are chosen to be answerable *only* from material a
text-only pipeline would have destroyed: a figure, a table, and a number buried in a table cell.


In [ ]:
QUESTIONS = [
    "Describe the overall encoder-decoder structure shown in the model architecture figure.",
    "What BLEU score did the big Transformer model achieve on English-to-German, and how does it "
    "compare to the previous best?",
    "How does the per-layer complexity of self-attention compare to that of a recurrent layer?",
]

for question in QUESTIONS:
    print("Q:", question)
    print("A:", chain.invoke(question))
    print("-" * 100)


In [ ]:
# What did the model actually receive? Never trust a RAG answer you have not audited.
question = QUESTIONS[0]
retrieved = retrieve(question)

print(f"Question: {question}\n")
for kind, original in retrieved:
    if kind == "image":
        print("[image] attached to the prompt:")
        display(Image(base64.b64decode(original)))
    else:
        print(f"[{kind}] {original[:300]}...\n")


### Reading the results honestly

Before you celebrate, check the obvious failure: **the Transformer paper is in the training data of
every model you can reach.** A plausible answer about the encoder stack proves nothing at all, because
the model could produce one with an empty context.

That is what the audit cell is for, and it is a habit rather than a one-off. Look at what was actually
retrieved. If the architecture figure is in there, the answer *can* be grounded. If it is not, and the
answer is still confident and correct, you have learned something important — your retrieval is not
working and your evaluation cannot tell.

The honest test for any RAG system is a question whose answer **cannot** be in the weights: your own
lecture slides, an internal report, a paper from last week. Swap `PDF_URL` for one of those and run the
notebook again. Everything except that one string is document-agnostic.


## Where this breaks

An affectionate list of the ways this pipeline will disappoint you, so that none of them are a surprise.

**The summary is a lossy proxy, and search only sees the proxy.** If the vision model does not mention
"positional encoding" while describing Figure 1, no question about positional encoding will ever
retrieve Figure 1. Your retrieval quality is capped by your summarisation prompt — which is why that
prompt asks for labels and numbers by name.

**Layout detection is a model, and models are wrong.** Figures get clipped, captions get orphaned into
their own text chunk, and two-column papers occasionally produce beautifully confident nonsense. Look
at the crops. Always look at the crops.

**Indexing is expensive; you pay it every time.** One LLM call per chunk, per table, per figure. Re-run
this notebook four times and you have summarised the same paper four times. Anything real caches
summaries keyed by a hash of the element, and re-summarises only what changed.

**The dict is not a database.** `docstore` dies with the kernel, and it holds every image in RAM as
base64, which is about 33% larger than the bytes. Fine for nine pages; not fine for nine hundred.

**Chunk-level retrieval is coarse.** A `by_title` chunk can be 8,000 characters, so a `k=4` retrieval
may drop 30,000 characters into the prompt. Cheaper and sharper: retrieve broadly, then re-rank, then
keep the top two.

**Tables inside images stay invisible.** If a table was scanned rather than typeset, `hi_res` finds an
`Image`, not a `Table`, and you get a vision-model *description* of numbers instead of the numbers.
Descriptions of numbers are not numbers. Do not compute with them.


## Play with it

**Break the summaries on purpose.** Change `IMAGE_PROMPT` to just "Describe this image briefly",
re-index, and ask the architecture question again. Does Figure 1 still get retrieved? This is the
fastest way to feel how much of multimodal RAG is really prompt engineering.

**Race it against text-only RAG.** Build a second chain over `texts` alone — no tables, no images — and
put the two side by side on the three questions above. Which questions separate them, and which ones do
both get right because the model simply knows the paper?

**Add a `kind` filter.** Chroma's `similarity_search` takes `filter={"kind": "table"}`. Wire that into
`retrieve()` so a question containing the word *table* searches only tables. Then find a question where
this helps, and one where it hurts.

**Go bigger and feel the cost.** Index all fifteen pages and count your LLM calls. Now imagine a
200-page manual, and work out what caching would have saved you.

**Bring your own PDF.** Point `PDF_URL` at something the model cannot possibly know — a course
handout, a report, last week's arXiv posting. This is the only test that really counts.

**Swap in real components.** Replace the dict with `langchain_core.stores.InMemoryStore`, then with
Redis; replace the hand-written `retrieve()` with `MultiVectorRetriever`. You now know exactly what
each of them does, which is the ideal moment to start using them.


## Glossary — quick reference

| Term | Meaning |
|---|---|
| **Multimodal RAG** | Retrieval where the indexed corpus contains images and tables, not only text. |
| **Multi-vector retrieval** | Search one representation (a summary), return a different one (the original), joined by an id. |
| **`partition_pdf`** | `unstructured`'s PDF reader: returns typed elements — `Table`, `Image`, `NarrativeText`, `Title`. |
| **`hi_res` strategy** | Render each page and run a layout-detection model over it. Slow; finds figures and tables. |
| **`orig_elements`** | The pre-chunking elements a `CompositeElement` swallowed. Where the images hide. |
| **`text_as_html`** | A table's structure preserved as real HTML, so columns survive the trip to the LLM. |
| **Data URI** | `data:image/jpeg;base64,...` — an image inlined into a request rather than linked. |
| **Content blocks** | A message body as a list of typed parts (`text`, `image_url`) instead of a plain string. |
| **`doc_id`** | The ticket number in a summary's metadata that points at the original on the shelf. |
| **Docstore** | Where originals live. A dict here; Redis, S3 or a database in anything real. |
| **Non-thinking mode** | `reasoning_effort="none"` — skip the reasoning tokens for tasks that do not need them. |
